
# Customer Segmentation — Compact Parquet-First Pipeline

This notebook starts from local parquet files only. It contains no SQL, Max/Min database access, credentials, or database connector code. The pipeline builds leakage-aware customer features, controls redundant features, handles extreme-value customers separately, compares K values, checks stability, and evaluates future outcomes.

**Required parquet files:** `seg_customers.parquet`, `seg_transactions.parquet`, `seg_balances.parquet`, `seg_products.parquet`, and optionally `seg_revenue_monthly.parquet`.


In [1]:

from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import (
    silhouette_score, calinski_harabasz_score, davies_bouldin_score,
    adjusted_rand_score
)
from scipy.stats import kruskal

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')
RANDOM_STATE = 42


In [2]:

# Parquet-first loader. Put parquet files in the notebook folder or its subfolders.
BASE_DIR = Path.cwd()
PARQUET_FILES = {
    'customers': 'seg_customers.parquet',
    'transactions': 'seg_transactions.parquet',
    'balances': 'seg_balances.parquet',
    'products': 'seg_products.parquet',
    'revenue': 'seg_revenue_monthly.parquet',
}

def find_file(filename):
    matches = list(BASE_DIR.rglob(filename))
    return matches[0] if matches else None

data = {}
missing = []
for name, filename in PARQUET_FILES.items():
    path = find_file(filename)
    if path is None:
        if name == 'revenue':
            print(f'Optional file not found: {filename}')
            data[name] = pd.DataFrame()
        else:
            missing.append(filename)
    else:
        data[name] = pd.read_parquet(path)
        print(f'Loaded {name}: {path} | shape={data[name].shape}')

if missing:
    raise FileNotFoundError(
        'Required parquet files are missing: ' + ', '.join(missing) +
        f'. Current folder: {BASE_DIR}'
    )

customers = data['customers'].copy()
transactions = data['transactions'].copy()
balances = data['balances'].copy()
products = data['products'].copy()
revenue = data['revenue'].copy()

for df, cols in [
    (customers, ['birth_date','signup_date']),
    (transactions, ['txn_ts','month']),
    (balances, ['month_start']),
    (products, ['opened_at']),
    (revenue, ['month_start'])
]:
    for col in cols:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors='coerce')

print('\nData quality summary:')
display(pd.DataFrame({
    'table': ['customers','transactions','balances','products','revenue'],
    'rows': [len(customers),len(transactions),len(balances),len(products),len(revenue)],
    'customer_ids': [
        customers['customer_id'].nunique() if 'customer_id' in customers else np.nan,
        transactions['customer_id'].nunique() if 'customer_id' in transactions else np.nan,
        balances['customer_id'].nunique() if 'customer_id' in balances else np.nan,
        products['customer_id'].nunique() if 'customer_id' in products else np.nan,
        revenue['customer_id'].nunique() if 'customer_id' in revenue else np.nan,
    ]
}))


Loaded customers: d:\ML\Uzcard\seg_customers.parquet | shape=(20000, 7)
Loaded transactions: d:\ML\Uzcard\seg_transactions.parquet | shape=(12136132, 8)
Loaded balances: d:\ML\Uzcard\seg_balances.parquet | shape=(427598, 3)
Loaded products: d:\ML\Uzcard\seg_products.parquet | shape=(29983, 3)
Loaded revenue: d:\ML\Uzcard\seg_revenue_monthly.parquet | shape=(427598, 7)

Data quality summary:


,table,rows,customer_ids
0,customers,20000,20000
1,transactions,12136132,19755
2,balances,427598,20000
3,products,29983,15844
4,revenue,427598,20000


In [3]:

# Observation windows. Features use only TRAIN_END and earlier; future outcomes use FUTURE_START onward.
TRAIN_END = pd.Timestamp('2025-09-30')
FUTURE_START = pd.Timestamp('2025-10-01')
FUTURE_END = pd.Timestamp('2025-12-31')
WINDOW_START = pd.Timestamp('2025-04-01')

# Use only columns available before the cutoff. No future revenue is used in clustering.


In [4]:

def entropy_from_counts(counts):
    p = counts.div(counts.sum(axis=1).replace(0, np.nan), axis=0).fillna(0)
    return -(p.where(p > 0, np.nan) * np.log(p.where(p > 0, np.nan))).sum(axis=1).fillna(0)

def build_features(cutoff, start=WINDOW_START, end=None):
    cutoff = pd.Timestamp(cutoff)
    end = pd.Timestamp(end or cutoff)
    ids = customers[['customer_id']].drop_duplicates().copy()
    tx = transactions.copy()
    tx = tx[(tx['txn_ts'] >= start) & (tx['txn_ts'] <= end)].copy()
    tx['amount_uzs'] = pd.to_numeric(tx['amount_uzs'], errors='coerce').fillna(0)
    tx['is_out'] = tx['direction'].astype(str).str.lower().eq('out') if 'direction' in tx else True
    out = tx[tx['is_out']].copy()

    if out.empty:
        features = ids.copy()
    else:
        out['month_key'] = out['txn_ts'].dt.to_period('M').astype(str)
        basic = out.groupby('customer_id').agg(
            txn_cnt=('txn_id','count'),
            total_spend=('amount_uzs','sum'),
            median_txn=('amount_uzs','median'),
            p90_txn=('amount_uzs', lambda x: x.quantile(.90)),
            txn_day_count=('txn_ts', lambda x: x.dt.normalize().nunique()),
            active_months=('month_key','nunique'),
            last_txn=('txn_ts','max'),
            first_txn=('txn_ts','min')
        ).reset_index()
        monthly = out.groupby(['customer_id','month_key']).size().rename('monthly_txn').reset_index()
        monthly_cv = monthly.groupby('customer_id')['monthly_txn'].agg(
            lambda x: float(x.std(ddof=0) / x.mean()) if x.mean() else 0.0
        ).rename('monthly_txn_cv').reset_index()
        basic = basic.merge(monthly_cv, on='customer_id', how='left')
        basic['log_total_spend'] = np.log1p(basic['total_spend'].clip(lower=0))
        basic['spend_per_active_month'] = basic['total_spend'] / basic['active_months'].clip(lower=1)
        basic['txn_per_active_month'] = basic['txn_cnt'] / basic['active_months'].clip(lower=1)
        n_months = max(1, len(pd.period_range(start, end, freq='M')))
        basic['active_month_ratio'] = basic['active_months'] / n_months
        basic['recency_days'] = (cutoff - basic['last_txn']).dt.days.clip(lower=0)
        basic['tenure_observed_days'] = (cutoff - basic['first_txn']).dt.days.clip(lower=0)
        basic['longest_gap_days'] = (basic['last_txn'] - basic['first_txn']).dt.days.clip(lower=0)

        channel_counts = out.groupby(['customer_id','channel'])['txn_id'].count().unstack(fill_value=0) if 'channel' in out else pd.DataFrame(index=basic['customer_id'])
        channel_shares = channel_counts.div(channel_counts.sum(axis=1), axis=0).fillna(0)
        channel_shares.columns = ['ch_share_' + str(c) for c in channel_shares.columns]
        channel_shares['channel_entropy'] = entropy_from_counts(channel_counts) if len(channel_counts.columns) else 0
        channel_shares['n_active_channels'] = (channel_counts > 0).sum(axis=1) if len(channel_counts.columns) else 0

        mcc_counts = out.groupby(['customer_id','mcc_group'])['txn_id'].count().unstack(fill_value=0) if 'mcc_group' in out else pd.DataFrame(index=basic['customer_id'])
        mcc_shares = mcc_counts.div(mcc_counts.sum(axis=1), axis=0).fillna(0)
        mcc_shares.columns = ['mcc_share_' + str(c) for c in mcc_shares.columns]
        mcc_shares['mcc_entropy'] = entropy_from_counts(mcc_counts) if len(mcc_counts.columns) else 0
        basic = basic.merge(channel_shares, on='customer_id', how='left').merge(mcc_shares, on='customer_id', how='left')
        features = ids.merge(basic, on='customer_id', how='left')

    # Profile/lifecycle features are retained for audit and optional experiments, not baseline clustering.
    profile_cols = [c for c in ['customer_id','birth_date','gender','region','signup_date'] if c in customers.columns]
    features = features.merge(customers[profile_cols].drop_duplicates('customer_id'), on='customer_id', how='left')
    if 'birth_date' in features:
        features['age_years'] = ((cutoff - features['birth_date']).dt.days / 365.25).clip(0, 110)
    if 'signup_date' in features:
        features['tenure_months'] = ((cutoff - features['signup_date']).dt.days / 30.4375).clip(lower=0)

    if not balances.empty:
        b = balances[(balances['month_start'] >= start) & (balances['month_start'] <= end)].copy()
        if 'avg_balance_uzs' in b:
            b['avg_balance_uzs'] = pd.to_numeric(b['avg_balance_uzs'], errors='coerce').fillna(0)
            bal = b.groupby('customer_id')['avg_balance_uzs'].agg(
                balance_median='median', balance_p90=lambda x: x.quantile(.90),
                balance_cv=lambda x: float(x.std(ddof=0) / x.mean()) if x.mean() else 0.0,
                positive_balance_months=lambda x: int((x > 0).sum())
            ).reset_index()
            features = features.merge(bal, on='customer_id', how='left')

    if not products.empty and 'product_code' in products:
        p = products[products['opened_at'] <= end].copy() if 'opened_at' in products else products.copy()
        prod = p.groupby('customer_id').agg(product_cnt=('product_code','nunique')).reset_index()
        features = features.merge(prod, on='customer_id', how='left')

    return features

features_train = build_features(TRAIN_END)
print('Training feature matrix:', features_train.shape)


Training feature matrix: (20000, 49)


In [5]:

# Compact, non-redundant baseline feature set.
# One representative variable is kept per latent concept; raw totals and labels are excluded.
BASE_FEATURES = [
    'recency_days', 'active_month_ratio', 'txn_per_active_month', 'monthly_txn_cv',
    'log_total_spend', 'median_txn', 'spend_per_active_month',
    'balance_median', 'balance_cv', 'positive_balance_months',
    'channel_entropy', 'n_active_channels', 'mcc_entropy', 'product_cnt', 'tenure_months'
]
BASE_FEATURES = [c for c in BASE_FEATURES if c in features_train.columns]
X_raw = features_train[BASE_FEATURES].replace([np.inf, -np.inf], np.nan).fillna(0).copy()

# Winsorize numeric tails so a few extreme customers do not define the entire geometry.
for col in X_raw.columns:
    lo, hi = X_raw[col].quantile([.01, .99])
    X_raw[col] = X_raw[col].clip(lo, hi)

scaler = StandardScaler()
X = pd.DataFrame(scaler.fit_transform(X_raw), columns=BASE_FEATURES, index=features_train.index).astype(np.float32)

corr = X.corr(method='spearman').abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
redundant_pairs = upper.stack().sort_values(ascending=False).reset_index()
redundant_pairs.columns = ['feature_a','feature_b','abs_spearman']
print('Baseline features:', BASE_FEATURES)
print('Highest correlations after reduction:')
display(redundant_pairs.head(12))


Baseline features: ['recency_days', 'active_month_ratio', 'txn_per_active_month', 'monthly_txn_cv', 'log_total_spend', 'median_txn', 'spend_per_active_month', 'balance_median', 'balance_cv', 'positive_balance_months', 'channel_entropy', 'n_active_channels', 'mcc_entropy', 'product_cnt', 'tenure_months']
Highest correlations after reduction:


,feature_a,feature_b,abs_spearman
0,log_total_spend,spend_per_active_month,0.9828
1,txn_per_active_month,spend_per_active_month,0.7781
2,txn_per_active_month,log_total_spend,0.7686
3,log_total_spend,mcc_entropy,0.6709
4,spend_per_active_month,mcc_entropy,0.6679
5,txn_per_active_month,mcc_entropy,0.6315
6,median_txn,spend_per_active_month,0.6155
7,log_total_spend,median_txn,0.6037
8,active_month_ratio,positive_balance_months,0.6033
9,spend_per_active_month,balance_median,0.5586


In [6]:

# Outlier/VIP review flag. These customers are not allowed to create a misleading ordinary cluster.
outlier_cols = [c for c in ['spend_per_active_month','balance_median','median_txn','log_total_spend'] if c in features_train]
outlier_flag = pd.Series(False, index=features_train.index)
for col in outlier_cols:
    outlier_flag |= features_train[col].rank(pct=True) >= .995
features_train['outlier_review'] = outlier_flag
print('Outlier-review customers:', int(outlier_flag.sum()))

# Main clustering population excludes the extreme-value review pocket.
cluster_mask = ~outlier_flag
X_cluster = X.loc[cluster_mask].reset_index(drop=True)
features_cluster = features_train.loc[cluster_mask].reset_index(drop=True).copy()
print('Customers used for ordinary clustering:', len(features_cluster))


Outlier-review customers: 271
Customers used for ordinary clustering: 19729


In [7]:

# Memory-safe K comparison: K-Means uses all ordinary customers; silhouette uses a fixed sample.
SIL_SAMPLE = min(2000, len(X_cluster))
rng = np.random.default_rng(RANDOM_STATE)
sil_idx = rng.choice(len(X_cluster), size=SIL_SAMPLE, replace=False)
X_sil = X_cluster.iloc[sil_idx]

rows = []
for k in range(2, 9):
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=30)
    labels = km.fit_predict(X_cluster)
    rows.append({
        'k': k,
        'inertia': float(km.inertia_),
        'silhouette_sample': float(silhouette_score(X_sil, labels[sil_idx])),
        'calinski_harabasz': float(calinski_harabasz_score(X_cluster, labels)),
        'davies_bouldin': float(davies_bouldin_score(X_cluster, labels)),
        'min_cluster_n': int(pd.Series(labels).value_counts().min()),
        'max_cluster_n': int(pd.Series(labels).value_counts().max()),
        'max_to_min_ratio': float(pd.Series(labels).value_counts().max() / pd.Series(labels).value_counts().min())
    })
k_metrics = pd.DataFrame(rows)
display(k_metrics)


  File "c:\Users\jasur\AppData\Local\Programs\Python\Python312\Lib\site-packages\joblib\externals\loky\backend\context.py", line 247, in _count_physical_cores
    cpu_count_physical = _count_physical_cores_win32()
                         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\jasur\AppData\Local\Programs\Python\Python312\Lib\site-packages\joblib\externals\loky\backend\context.py", line 299, in _count_physical_cores_win32
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "c:\Users\jasur\AppData\Local\Programs\Python\Python312\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\jasur\AppData\Local\Programs\Python\Python312\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "c:\Users\jasur\AppData\Local\Programs\Python\Python312\Lib\subprocess.py", line 1538, in _execute_child
    hp, ht, pid, tid = _winapi.Crea

,k,inertia,silhouette_sample,calinski_harabasz,davies_bouldin,min_cluster_n,max_cluster_n,max_to_min_ratio
0,2,"220,386.9375",0.5712,"5,973.6890",0.9996,1078,18651,17.3015
1,3,"191,584.5938",0.2506,"4,918.4668",1.6470,1021,14496,14.1978
2,4,"168,049.8594",0.2677,"4,658.7988",1.4634,844,13926,16.5000
3,5,"152,436.7031",0.2352,"4,356.8379",1.3885,828,12148,14.6715
4,6,"137,701.9688",0.2495,"4,280.3218",1.2865,442,12151,27.4910
5,7,"127,083.7812",0.1711,"4,139.3945",1.4497,440,8976,20.4000
6,8,"119,930.2969",0.1493,"3,927.5417",1.5040,440,5309,12.0659


In [10]:
K_FINAL = 4
final_model = KMeans(n_clusters=K_FINAL, random_state=RANDOM_STATE, n_init=50)
ordinary_labels = final_model.fit_predict(X_cluster)

# 2. Convert integer labels to strings to avoid dtype mixing
string_labels = [f"cluster_{label}" for label in ordinary_labels]
features_cluster['cluster'] = string_labels 

# 3. Add the outlier review pocket as a non-ordinary state
features_train['cluster'] = 'ordinary_unassigned'
features_train.loc[cluster_mask, 'cluster'] = string_labels  # Safe string assignment
features_train.loc[outlier_flag, 'cluster'] = 'outlier_review'

# 4. Aggregation and display
cluster_counts = features_train['cluster'].value_counts().rename_axis('cluster').to_frame('n_customers')
cluster_counts['share'] = cluster_counts['n_customers'] / len(features_train)

display(cluster_counts)

,n_customers,share
cluster,,
cluster_3,13926,0.6963
cluster_0,4082,0.2041
cluster_2,877,0.0439
cluster_1,844,0.0422
outlier_review,271,0.0135


In [11]:

# Profile table: report medians and sample sizes, not only means.
profile_cols = [c for c in BASE_FEATURES if c in features_train.columns]
profile_median = features_train.groupby('cluster')[profile_cols].median(numeric_only=True).T
display(profile_median)


cluster,cluster_0,cluster_1,cluster_2,cluster_3,outlier_review
recency_days,1.0000,1.0000,35.0000,2.0000,1.0000
active_month_ratio,1.0000,0.3333,0.3333,1.0000,1.0000
txn_per_active_month,30.3667,16.0000,1.0000,14.1667,12.1667
monthly_txn_cv,0.3324,0.2000,0.0000,0.3356,0.3409
log_total_spend,18.4910,16.2678,12.5505,16.8446,18.5933
median_txn,"601,067.5000","192,426.0000","112,343.0000","154,859.5000","927,969.0000"
spend_per_active_month,"18,002,607.3333","5,320,179.2500","128,559.3333","3,482,514.5833","20,225,137.5000"
balance_median,"9,630,894.5000","1,803,890.0000","303,149.5000","1,114,885.5000","27,769,146.5000"
balance_cv,0.2963,0.1477,0.2827,0.2944,0.2829
positive_balance_months,6.0000,2.0000,6.0000,6.0000,6.0000


In [12]:

# Stability audit: compare repeated seeds and half-samples on ordinary customers.
base_labels = ordinary_labels
stability_rows = []
for seed in [0, 1, 7, 19, 42, 99]:
    labels = KMeans(n_clusters=K_FINAL, random_state=seed, n_init=30).fit_predict(X_cluster)
    stability_rows.append({'test':'seed','seed':seed,'ari_vs_base':adjusted_rand_score(base_labels, labels)})
for seed in range(5):
    sample_idx = np.random.default_rng(seed).choice(len(X_cluster), size=max(1000, len(X_cluster)//2), replace=False)
    sample_labels = KMeans(n_clusters=K_FINAL, random_state=RANDOM_STATE, n_init=30).fit_predict(X_cluster.iloc[sample_idx])
    stability_rows.append({'test':'half_sample','seed':seed,'ari_vs_base':adjusted_rand_score(base_labels[sample_idx], sample_labels)})
stability = pd.DataFrame(stability_rows)
display(stability)


,test,seed,ari_vs_base
0,seed,0,0.9996
1,seed,1,0.9996
2,seed,7,0.9996
3,seed,19,0.9996
4,seed,42,1.0000
5,seed,99,0.9996
6,half_sample,0,0.9868
7,half_sample,1,0.9687
8,half_sample,2,0.9785
9,half_sample,3,0.9929


In [13]:

# Leakage-safe future validation. Clusters were formed only through TRAIN_END.
def future_outcomes(start=FUTURE_START, end=FUTURE_END):
    tx = transactions[(transactions['txn_ts'] >= start) & (transactions['txn_ts'] <= end)].copy()
    tx['amount_uzs'] = pd.to_numeric(tx['amount_uzs'], errors='coerce').fillna(0)
    tx_out = tx[tx['direction'].astype(str).str.lower().eq('out')] if 'direction' in tx else tx
    future_tx = tx_out.groupby('customer_id').agg(
        future_txn_count=('txn_id','count'),
        future_spend=('amount_uzs','sum')
    ).reset_index()
    result = customers[['customer_id']].drop_duplicates().merge(future_tx, on='customer_id', how='left').fillna(0)
    result['future_active'] = (result['future_txn_count'] > 0).astype(int)
    if not revenue.empty and 'net_revenue_uzs' in revenue:
        rv = revenue[(revenue['month_start'] >= start) & (revenue['month_start'] <= end)]
        future_rev = rv.groupby('customer_id')['net_revenue_uzs'].sum().rename('future_net_revenue').reset_index()
        result = result.merge(future_rev, on='customer_id', how='left').fillna(0)
    else:
        result['future_net_revenue'] = np.nan
    return result

future = future_outcomes()
external = features_train[['customer_id','cluster']].merge(future, on='customer_id', how='left')
external_summary = external.groupby('cluster').agg(
    n_customers=('customer_id','count'),
    future_active_rate=('future_active','mean'),
    median_future_txn=('future_txn_count','median'),
    mean_future_txn=('future_txn_count','mean'),
    median_future_spend=('future_spend','median'),
    mean_future_spend=('future_spend','mean'),
    median_future_net_revenue=('future_net_revenue','median')
).sort_index()
display(external_summary)


,n_customers,future_active_rate,median_future_txn,mean_future_txn,median_future_spend,mean_future_spend,median_future_net_revenue
cluster,,,,,,,
cluster_0,4082,1.0000,90.0000,121.8111,"56,362,366.5000","86,314,064.7660","633,672.5000"
cluster_1,844,1.0000,49.0000,82.3306,"16,421,427.5000","31,088,570.6055","145,781.0000"
cluster_2,877,0.5245,1.0000,2.7389,"37,908.0000","748,890.1631","5,821.0000"
cluster_3,13926,0.9989,46.0000,68.3705,"11,202,011.0000","14,895,881.5447","86,062.0000"
outlier_review,271,1.0000,45.0000,168.5793,"71,011,018.0000","453,394,521.1661","1,587,707.0000"



## Interpretation rules

The ordinary clusters are not treated as ground truth. Pairwise classifier accuracy is not reported as segmentation quality because it only reconstructs labels created by the same feature space. The final decision must consider minimum cluster size, sample silhouette, redundancy, stability, future outcome separation, and whether each group has a defensible business action. The `outlier_review` state is reported separately and is not presented as a normal customer persona.
